Tabelle mit error berechnen

In [1]:
import math

def calc_error_UI(U_I):
    val = abs(U_I)
    err_read = 0.0003  # Ablesefehler 0.3 mV = 0.0003 V
    
    # Bestimmung des Messbereichs und Gerätefehlers laut Foto
    if val < 0.4:
        # 400 mV Bereich: 0.25% + 5 Digit (Auflösung 0.1 mV = 0.0001 V)
        err_device = 0.0025 * val + 5 * 0.0001
    elif val < 4.0:
        # 4 V Bereich: 0.4% + 1 Digit (Auflösung 1 mV = 0.001 V)
        err_device = 0.004 * val + 1 * 0.001
    else:
        # 40 V Bereich: 0.25% + 1 Digit (Auflösung 10 mV = 0.01 V)
        err_device = 0.0025 * val + 1 * 0.01
        
    return err_device + err_read

def to_latex_val(val, err, decimals=3):
    if val is None:
        return "-"
    # Formatiert als "Wert \pm Fehler" und ersetzt Punkte durch Kommas
    val_str = f"{val:.{decimals}f}".replace('.', ',')
    err_str = f"{err:.{decimals}f}".replace('.', ',')
    return f"${val_str} \\pm {err_str}$"

def generate_latex_table(name, u_vals, ui_vals, ui0):
    latex_out = f"% === Tabelle: {name} ===\n"
    
    err_ui0 = calc_error_UI(ui0)
    
    row_ui_ui0 = []
    row_err_diff = []
    row_sqrt = []
    row_err_sqrt = []
    
    for ui in ui_vals:
        err_ui = calc_error_UI(ui)
        diff = ui - ui0
        # Gaußsche Fehlerfortpflanzung für die Differenz
        err_diff = math.sqrt(err_ui**2 + err_ui0**2)
        
        row_ui_ui0.append(diff)
        row_err_diff.append(err_diff)
        
        if diff >= 0:
            s_val = math.sqrt(diff)
            # Fehlerfortpflanzung Wurzel: Delta(sqrt(x)) = Delta(x) / (2*sqrt(x))
            s_err = err_diff / (2 * s_val) if s_val > 0 else 0
            row_sqrt.append(s_val)
            row_err_sqrt.append(s_err)
        else:
            row_sqrt.append(None)
            row_err_sqrt.append(None)
            
    # Aufteilen in kleinere Blöcke, da \pm viel Platz braucht (max 6 Spalten)
    chunk_size = 6
    for i in range(0, len(u_vals), chunk_size):
        u_chunk = u_vals[i:i+chunk_size]
        ui_chunk = ui_vals[i:i+chunk_size]
        
        diff_chunk = row_ui_ui0[i:i+chunk_size]
        err_diff_chunk = row_err_diff[i:i+chunk_size]
        
        sqrt_chunk = row_sqrt[i:i+chunk_size]
        err_sqrt_chunk = row_err_sqrt[i:i+chunk_size]
        
        cols = len(u_chunk)
        latex_out += "\\begin{table}[H]\n\\centering\n"
        if len(u_vals) > chunk_size:
            latex_out += f"\\caption{{Messwerte {name} (Teil {i//chunk_size + 1})}}\n"
        else:
            latex_out += f"\\caption{{Messwerte {name}}}\n"
            
        latex_out += "\\resizebox{\\textwidth}{!}{\n"
        latex_out += "\\begin{tabular}{|c|" + "c|" * cols + "}\n\\hline\n"
        
        # U
        latex_out += " $U\\,[\\text{V}]$ & " + " & ".join([f"${x:.1f}$".replace('.', ',') for x in u_chunk]) + " \\\\\\hline\n"
        # U_I
        latex_out += " $U_I\\,[\\text{V}]$ & " + " & ".join([to_latex_val(ui_chunk[j], calc_error_UI(ui_chunk[j])) for j in range(cols)]) + " \\\\\\hline\n"
        # U_I - U_I0
        latex_out += " $U_I - U_{I0}\\,[\\text{V}]$ & " + " & ".join([to_latex_val(diff_chunk[j], err_diff_chunk[j]) for j in range(cols)]) + " \\\\\\hline\n"
        # Sqrt
        latex_out += " $\\sqrt{U_I - U_{I0}}$ & " + " & ".join([to_latex_val(sqrt_chunk[j], err_sqrt_chunk[j]) for j in range(cols)]) + " \\\\\\hline\n"
        
        latex_out += "\\end{tabular}\n}\n\\end{table}\n\n"
        
    return latex_out


# --- DATEN AUS DEM MESSPROTOKOLL ---

# 1. UV-Linie (365 nm)
u_uv = [0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0, -1.1, -1.2, -1.3, -1.4, -1.5, -1.6, -1.7, -1.8, -1.9]
ui_uv = [4.07, 3.69, 3.37, 3.03, 2.737, 2.435, 2.143, 1.861, 1.619, 1.386, 1.176, 0.985, 0.811, 0.640, 0.479, 0.3287, 0.1824, 0.0838, 0.0130, -0.0130]
print(generate_latex_table("UV-Linie ($\\lambda = 365\\,\\text{nm}$)", u_uv, ui_uv, -0.032))

# 2. Violette Linie (405 nm)
u_vi = [0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0, -1.1, -1.2, -1.3, -1.4, -1.5, -1.6]
ui_vi = [4.690, 4.33, 3.94, 3.58, 3.190, 2.853, 2.482, 2.182, 1.862, 1.584, 1.3250, 1.0890, 0.8720, 0.6860, 0.5010, 0.3223, 0.1648, 0.0662, 0.0056, -0.0136]
print(generate_latex_table("Violette Linie ($\\lambda = 405\\,\\text{nm}$)", u_vi, ui_vi, -0.0267))

# 3. Blaue Linie (436 nm)
u_bl = [0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0, -1.1, -1.2, -1.3, -1.4]
ui_bl = [6.08, 5.58, 5.04, 4.56, 3.99, 3.544, 3.097, 2.654, 2.252, 1.840, 1.479, 1.175, 0.870, 0.577, 0.3103, 0.1188, 0.0127, -0.0208]
print(generate_latex_table("Blaue Linie ($\\lambda = 436\\,\\text{nm}$)", u_bl, ui_bl, -0.0393))

# 4. Grüne Linie (546 nm)
u_gr = [0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0]
ui_gr = [3.049, 2.530, 2.033, 1.547, 1.132, 0.776, 0.477, 0.266, 0.1196, 0.0474, 0.0165, 0.0060, 0.0001, -0.0046]
print(generate_latex_table("Grüne Linie ($\\lambda = 546\\,\\text{nm}$)", u_gr, ui_gr, -0.0162))

# 5. Gelbe Linie (578 nm)
u_ye = [0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0, -1.1, -1.2]
ui_ye = [0.584, 0.405, 0.2622, 0.1640, 0.0985, 0.0660, 0.0475, 0.0350, 0.0280, 0.0217, 0.0163, 0.0115, 0.0077, 0.0043, 0.0014, -0.0010]
print(generate_latex_table("Gelbe Linie ($\\lambda = 578\\,\\text{nm}$)", u_ye, ui_ye, -0.007))

% === Tabelle: UV-Linie ($\lambda = 365\,\text{nm}$) ===
\begin{table}[H]
\centering
\caption{Messwerte UV-Linie ($\lambda = 365\,\text{nm}$) (Teil 1)}
\resizebox{\textwidth}{!}{
\begin{tabular}{|c|c|c|c|c|c|c|}
\hline
 $U\,[\text{V}]$ & $0,0$ & $-0,1$ & $-0,2$ & $-0,3$ & $-0,4$ & $-0,5$ \\\hline
 $U_I\,[\text{V}]$ & $4,070 \pm 0,020$ & $3,690 \pm 0,016$ & $3,370 \pm 0,015$ & $3,030 \pm 0,013$ & $2,737 \pm 0,012$ & $2,435 \pm 0,011$ \\\hline
 $U_I - U_{I0}\,[\text{V}]$ & $4,102 \pm 0,020$ & $3,722 \pm 0,016$ & $3,402 \pm 0,015$ & $3,062 \pm 0,013$ & $2,769 \pm 0,012$ & $2,467 \pm 0,011$ \\\hline
 $\sqrt{U_I - U_{I0}}$ & $2,025 \pm 0,005$ & $1,929 \pm 0,004$ & $1,844 \pm 0,004$ & $1,750 \pm 0,004$ & $1,664 \pm 0,004$ & $1,571 \pm 0,004$ \\\hline
\end{tabular}
}
\end{table}

\begin{table}[H]
\centering
\caption{Messwerte UV-Linie ($\lambda = 365\,\text{nm}$) (Teil 2)}
\resizebox{\textwidth}{!}{
\begin{tabular}{|c|c|c|c|c|c|c|}
\hline
 $U\,[\text{V}]$ & $-0,6$ & $-0,7$ & $-0,8$ & $-0,9$ &